# Phase 9 — Baseline Machine Learning (readiness gate)

## Why this notebook does not fit a model yet

Baseline ML is appropriate **only after** the target represents the intended outcome and the split matches deployment. Phase 8 found that this dataset does not currently support reliable measured RUL labels: the aging log has failure-like notes for Devices 3 and 5 but no uniform EoL rule for all four devices, Devices 2 and 4 have no explicit failure entry, and MAT `date` text disagrees with `timeEpoch` interpreted as Unix time by decades. The existing `remaining_observed_hours` target means time to the end of the recorded segment, not time to verified physical failure.

Therefore, this is a **readiness and leakage lesson**, not a model-comparison exercise. We will not train Linear Regression, Random Forest, Gradient Boosting, or XGBoost against the observation-horizon proxy. MAE/RMSE on that target would quantify proxy prediction only and could misleadingly be presented as RUL accuracy. Resolve and document BoL, EoL, failure criteria, event-time alignment, and censoring first.

## 1. What data leakage means

**Data leakage** occurs when information unavailable at the intended prediction time influences training, feature construction, model selection, or evaluation. The test score then benefits from a shortcut that would not exist in real deployment, so it overstates future performance.

In run-to-failure PHM, leakage can be subtle:

- **Randomly splitting rows:** nearby captures from the same device/run land in both train and test. Strong device-specific and temporal similarity makes test rows look familiar.
- **Using future-derived features:** full-run normalization, smoothing across the cutoff, or statistics calculated using later captures let future behavior influence a present-time prediction.
- **Using target construction as a predictor:** remaining time, EoL timestamp, final-record index, or features computed from future-to-failure are direct answers or proxies for the target.
- **Preprocessing before splitting:** fitting scalers, imputers, feature selection, or dimensionality reduction on all data lets the test distribution shape the training pipeline.
- **Overlapping windows across a boundary:** windows that share raw samples can place almost the same signal in train and test.
- **Tuning on the test set:** selecting features, model families, or hyperparameters repeatedly based on the final test score makes that test set part of model development.

Prevent leakage by defining the prediction time and deployment population first, constructing each feature only from observations available by that time, splitting before fitted preprocessing, and preserving an untouched final test group/period.

## 2. Choose a split that matches the PHM deployment question

There are at least two different generalization questions:

1. **Future prediction for already monitored devices:** choose a chronological cutoff within each device. Train only on times before the cutoff; evaluate only on later times. If multiple devices are available, use device-grouped development folds and reserve a later period or a full device for final evaluation.
2. **Prediction for a previously unseen device:** hold out entire devices (`GroupKFold` / leave-one-device-out). All records from a test device remain outside training. If the goal is also future forecasting, additionally respect chronology within the held-out device and do not use future observations to build its inputs.

A chronological split within one device answers a different question from leave-one-device-out. Both may be useful, but neither repairs invalid targets or confounded measurements. Split at the unit/run level before scaling or fitting transformations. For censored run-to-failure data, ordinary supervised regression also needs a target/evaluation design that handles censoring rather than dropping or mislabeling censored devices.

## 3. Models and metrics—when the target is ready

When a verified RUL target and adequate cohort exist, begin with simple references:

- **Linear Regression:** transparent baseline; inspect residuals and whether the relationship is plausibly linear.
- **Random Forest:** captures nonlinear interactions, but can predict poorly outside the training feature/time range and does not inherently model temporal dynamics.
- **Gradient Boosting:** may improve tabular fit, but tune only within training folds. XGBoost is optional if its added complexity is justified and the project has the dependency.

Use **MAE** (typical absolute error) and **RMSE** (penalizes large errors more) in the target's actual unit, e.g. verified hours-to-EoL. Report per-device/run scores and an appropriate aggregate; a pooled row-level score can overweight devices with more captures. Compare with a naive baseline, such as the training-set median RUL, and report uncertainty across held-out devices/runs. Do not compare models on different target definitions or leakage-prone splits.

These metrics are not computed here because the current target is not verified RUL.

## 4. Dataset readiness decision

| Requirement | Status in this project | Consequence |
|---|---|---|
| Defensible BoL / exposure coordinate | Test-start notes exist, but no uniform healthy baseline or complete cycles/dose timeline. | Cannot claim elapsed recorded time equals aging dose. |
| Uniform failure/EoL definition | Not established for the selected four-device experiment. | No consistent target endpoint. |
| Verified EoL label per device | Failure-like notes for Devices 3 and 5; none explicitly recorded for 2 and 4. | Mixed events/censoring cannot be treated as four exact failure times. |
| Synchronized event and measurement time | MAT date strings conflict with `timeEpoch` interpreted as Unix seconds. | Cannot safely calculate time-to-event from those fields. |
| Validated feature/measurement pipeline | Transient descriptors are exploratory; current drift and changing conditions are documented. | A model can learn acquisition/stress signatures instead of health. |
| Adequate independent validation units | Four devices only. | Model comparisons/generalization estimates would be highly uncertain. |

**Decision: not ready to train an RUL model.** Do not train on `data/features/rul_features.csv` as if its `remaining_observed_hours` were physical RUL. That file may be used only to demonstrate a remaining-observation-horizon proxy, with that label stated prominently. Once event definitions and timestamps are resolved, construct a new target table with explicit status fields (failure observed vs censored, event-time source, time coordinate, assumptions) and design validation around deployment.

## Phase 9 learning check

1. Why is a random row split unsafe for repeated time-series captures from the same device?
2. Which split answers future prediction on known devices, and which tests transfer to unseen devices?
3. Give an example of preprocessing leakage and how to prevent it.
4. What would MAE and RMSE mean if the labels were only time-to-last-record?
5. Why can a high test score not validate an undefined or assumed EoL?
6. What evidence must be resolved before fitting and comparing RUL models here?

No model is fitted in this phase because the Phase 8 readiness conditions are unmet. This is the correct PHM decision—not a missing implementation.